Builds every manuscript table, figure, confidence interval and the methods statement from the
SAVED results of the single versioned run. Nothing is recomputed from MIMIC and no random numbers
are drawn here: all intervals come from paired_dkappa_all_variants.csv.

In [ ]:

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
PROCESSED = Path.cwd() / "processed_outputs"
SENS = PROCESSED / "Sensitivity_Analysis"
OUT = PROCESSED / "Tables_and_Figures"
D_TAB, D_FIG, D_ADD, D_TXT = OUT / "tables", OUT / "figures", OUT / "figures_additional", OUT / "text"
for d_ in (D_TAB, D_FIG, D_ADD, D_TXT):
    d_.mkdir(parents=True, exist_ok=True)

In [ ]:
INPUTS = {
    "registry": PROCESSED / "variant_registry.csv",
    "paired": PROCESSED / "paired_dkappa_all_variants.csv",
    "summary": PROCESSED / "phase2_final_summary.csv",
    "gapstats": PROCESSED / "gap_sweep_full_stats.csv",
    "cohort": PROCESSED / "cohort.parquet",
    "frozen": PROCESSED / "labels_v5.parquet",
    "runlog": SENS / "run_log_sensitivity.json",
}
for k, p in INPUTS.items():
    if not p.exists():
        raise FileNotFoundError(f"missing input {p}. Run the sensitivity notebook first.")
FLAGS_PATH = PROCESSED / "stay_flags.parquet"

In [ ]:
reg = pd.read_csv(INPUTS["registry"])
pa = pd.read_csv(INPUTS["paired"])
p2 = pd.read_csv(INPUTS["summary"])
gs = pd.read_csv(INPUTS["gapstats"])
cohort = pd.read_parquet(INPUTS["cohort"])
fz = pd.read_parquet(INPUTS["frozen"])
flags = pd.read_parquet(FLAGS_PATH) if FLAGS_PATH.exists() else None
runlog = json.loads(INPUTS["runlog"].read_text())
SEED, N_BOOT = runlog["seed"], runlog["n_boot"]

#### Consistency check (every file must describe the same run)

In [ ]:
assert len(reg) == 24 and reg["variant"].is_unique, "registry must have 24 unique rows"
assert set(reg["variant"]) == set(p2["Variant"]), "registry and summary list different variants"
assert set(pa["Variant"]) == set(reg["variant"]) - {"baseline"}, "paired file must hold the 23 variants"
m = reg.merge(p2, left_on="variant", right_on="Variant")
assert (m["cohort_n"] == m["Cohort N"]).all(), "cohort N differs between registry and summary"
assert (m["early_excluded_n"] == m["Early-Excluded N"]).all(), "early-excluded N differs"
assert (m["labels_changed_common"] == m["Labels Changed"]).all(), "labels changed differs"
assert (m["common_stays_n"] == m["Common w/ baseline N"]).all(), "common stays differ"
mp = reg.merge(pa, left_on="variant", right_on="Variant")
assert (mp["common_with_reference_n"] == mp["combined n"]).all(), "bootstrap n != registry denominator"
assert (mp["labels_changed_common"] == mp["Labels changed"]).all(), "labels changed differs (paired)"
base = reg.loc[reg["variant"] == "baseline"].iloc[0]
assert int(base["cohort_n"]) == len(fz) == int(p2.loc[p2["Variant"] == "baseline", "Cohort N"].iloc[0])
assert int(fz["AKI"].sum()) == int(p2.loc[p2["Variant"] == "baseline", "AKI N"].iloc[0])
print("consistency checks passed")

In [ ]:
NODATA = "-"


def vname(v):
    return "V5" if v == "baseline" else v


def fmt_n(x):
    return f"{int(x):,}"


def fmt_ci(o, lo, hi):
    """+0.0000 [+0.0000, +0.0000]*, with * when the interval excludes 0.
    """
    if o == 0 and lo == 0 and hi == 0:
        return NODATA
    star = "*" if (lo > 0 or hi < 0) else ""
    return f"{o + 0.0:+.4f} [{lo + 0.0:+.4f}, {hi + 0.0:+.4f}]{star}"


def pct(a, b, d=2):
    return round(100 * a / b, d) if b else np.nan

#### Shared numbers

In [ ]:
N_total = len(cohort)
n_early = int(base["early_excluded_n"])
n_final = int(base["cohort_n"])
n_after_early = N_total - n_early
n_noev = n_after_early - n_final
n_aki = int(fz["AKI"].sum())
n_ref = int(base["common_with_reference_n"])
b2 = p2.loc[p2["Variant"] == "baseline"].iloc[0]
g = gs.loc[gs["label"] == "baseline"].iloc[0]
checks = []# (description, passed) written to the consistency report



def check(desc, ok):
    checks.append((desc, bool(ok)))
    if not ok:
        print("CONSISTENCY FAILURE:", desc)

#### Table 1 - baseline characteristics of the analysis cohort

In [ ]:
ID = "stay_id"
cohort_f = cohort.merge(fz[[ID, "AKI"]], on=ID, how="inner")
if flags is not None:
    extra = [c for c in flags.columns if c != ID and c not in cohort_f.columns]
    cohort_f = cohort_f.merge(flags[[ID] + extra], on=ID, how="left")
# fall back to the ESRD flag stored with the frozen labels
if "has_esrd" in fz.columns and not any(c in cohort_f.columns for c in ("has_esrd", "esrd")):
    cohort_f = cohort_f.merge(fz[[ID, "has_esrd"]], on=ID, how="left")
check("analysis cohort rows match labels_v5", len(cohort_f) == n_final)
SEX = {"M": "Male", "F": "Female"}
# key: (candidate columns, label, kind)
CAND = [
    ("age", ["anchor_age", "age"], "Age, years", "cont"),
    ("sex", ["gender", "sex"], "Sex", "cat"),
    ("race", ["race", "race_group"], "Race / ethnicity", "cat"),
    ("adm", ["admission_type"], "Admission type", "cat"),
    ("unit", ["first_careunit", "careunit"], "First ICU care unit", "cat"),
    ("los", ["los", "los_icu"], "ICU length of stay, days", "cont"),
    ("los_h", ["los_hours", "icu_los_hours"], "ICU length of stay, hours", "cont"),
    ("esrd", ["has_esrd", "esrd"], "ESRD or chronic dialysis", "bin"),
    ("death", ["hospital_expire_flag", "in_hospital_death"], "In-hospital death", "bin"),
    ("base", ["baseline_scr", "baseline_creatinine"], "Baseline creatinine, mg/dL", "cont"),
    ("wt", ["weight_kg", "weight"], "Weight, kg", "cont"),
]
found, missing = [], []
for key, names, label, kind in CAND:
    col = next((n for n in names if n in cohort_f.columns), None)
    if col is not None:
        found.append((key, col, label, kind))
    elif key != "los_h": # hours variant is only a fallback for days
        missing.append(label)
if any(k == "los" for k, *_ in found):
    found = [f for f in found if f[0] != "los_h"]
    missing = [x for x in missing if x != "ICU length of stay, hours"]
if not any(k == "los" for k, *_ in found):
    missing = [x for x in missing if x != "ICU length of stay, days"]

groups = [("All analysis stays", cohort_f), ("AKI-negative", cohort_f[~cohort_f["AKI"].astype(bool)]),
          ("AKI-positive", cohort_f[cohort_f["AKI"].astype(bool)])]




In [ ]:
def cont_cell(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return NODATA
    q1, med, q3 = s.quantile([0.25, 0.5, 0.75])
    return f"{med:.1f} [{q1:.1f}, {q3:.1f}]"




In [ ]:
def cat_levels(series, top=7):
    s = series.fillna("Missing").astype(str)
    lv = s.value_counts().index.tolist()
    return s, (lv[:top] if len(lv) > top + 1 else lv)




In [ ]:
t1_rows = [["N (stays)"] + [fmt_n(len(d)) for _, d in groups]]
for key, col, label, kind in found:
    if kind == "cont":
        t1_rows.append([f"{label}, median [IQR]"] + [cont_cell(d[col]) for _, d in groups])
        n_miss = [int(pd.to_numeric(d[col], errors="coerce").isna().sum()) for _, d in groups]
        if any(n_miss):
            t1_rows.append(["  missing, n (%)"] + [f"{a:,} ({pct(a, len(d), 1):.1f}%)" for a, (_, d) in zip(n_miss, groups)])
    elif kind == "bin":
        cells = []
        for _, d in groups:
            v = pd.to_numeric(d[col], errors="coerce").fillna(0).astype(bool)
            cells.append(f"{int(v.sum()):,} ({pct(int(v.sum()), len(d), 1):.1f}%)")
        t1_rows.append([label + ", n (%)"] + cells)
    else:
        t1_rows.append([label + ", n (%)"] + [""] * 3)
        allv, levels = cat_levels(cohort_f[col])
        if key == "sex":
            levels = sorted(levels, key=lambda x: (x != "M", x))
        shown = cohort_f[col].fillna("Missing").astype(str)
        for lv in levels:
            cells = []
            for _, d in groups:
                v = d[col].fillna("Missing").astype(str)
                c = int((v == lv).sum())
                cells.append(f"{c:,} ({pct(c, len(d), 1):.1f}%)")
            t1_rows.append([f"  {SEX.get(lv, lv) if key == 'sex' else lv}"] + cells)
        rest = [lv for lv in allv.unique() if lv not in levels]
        if rest:
            cells = []
            for _, d in groups:
                v = d[col].fillna("Missing").astype(str)
                c = int(v.isin(rest).sum())
                cells.append(f"{c:,} ({pct(c, len(d), 1):.1f}%)")
            t1_rows.append(["  Other"] + cells)
t1 = pd.DataFrame(t1_rows, columns=["Characteristic", f"All analysis stays (N = {n_final:,})",
                                    f"AKI-negative (N = {n_final - n_aki:,})", f"AKI-positive (N = {n_aki:,})"])


In [ ]:
check("Table 1 group sizes add up", len(groups[1][1]) + len(groups[2][1]) == n_final)
check("Table 1 AKI-positive N equals labels_v5 AKI sum", len(groups[2][1]) == n_aki)
for key, col, label, kind in found:
    if kind == "cat": # every categorical block must add up to the cohort
        _, lv = cat_levels(cohort_f[col])
        rows = t1_rows
        i0 = next(i for i, r in enumerate(rows) if r[0] == label + ", n (%)")
        tot = 0
        for r in rows[i0 + 1:]:
            if not r[0].startswith("  ") or not r[1]:
                break
            tot += int(r[1].split(" ")[0].replace(",", ""))
        check(f"Table 1 '{label}' levels add up to the cohort", tot == n_final)


#### Table 2 - variant comparison

In [ ]:
reg_i = reg.set_index("variant")
pa_i = pa.set_index("Variant")
p2_i = p2.set_index("Variant")
VARIANTS = [v for v in reg["variant"] if v != "baseline"] # the 23 variants, registry order
chg_pct = 100 * reg_i["labels_changed_common"] / reg_i["common_stays_n"]
# Table 3 and Figure 2: every variant that changes at least one label (a paired change exists only then)
T3_SET = [v for v in VARIANTS if reg_i.loc[v, "labels_changed_common"] > 0]


def grp(v):
    r = pa_i.loc[v]
    return 0 if r["dkappa urine"] != 0 else (1 if r["dkappa creatinine"] != 0 else 2)


ORDER = sorted(T3_SET, key=lambda v: (grp(v), -pa_i.loc[v, "dkappa urine"] if grp(v) == 0 else
                                       (-pa_i.loc[v, "dkappa creatinine"] if grp(v) == 1 else -pa_i.loc[v, "dkappa combined"])))


In [ ]:
T2_ORDER = sorted(VARIANTS, key=lambda v: -int(reg_i.loc[v, "labels_changed_common"]))
t2_rows = []
for v in ["baseline"] + T2_ORDER:
    r, s = reg_i.loc[v], p2_i.loc[v]
    if v == "baseline":
        ch, dk = NODATA, NODATA
    else:
        c_, cs_ = int(r["labels_changed_common"]), int(r["common_stays_n"])
        ch = f"{c_:,} ({pct(c_, cs_, 2):.2f}%)"
        q = pa_i.loc[v]
        dk = fmt_ci(q["dkappa combined"], q["combined CI low"], q["combined CI high"])
    t2_rows.append({"Variant": vname(v), "Cohort N": fmt_n(r["cohort_n"]), "Early-excl. N": fmt_n(r["early_excluded_n"]),
                    "Evaluable N": fmt_n(s["Evaluable N"]), "AKI N": fmt_n(s["AKI N"]), "AKI %": f"{float(s['AKI %']):.2f}",
                    "Common N": fmt_n(r["common_stays_n"]), "Labels changed N (%)": ch,
                    "Kappa vs reference (n)": f"{float(s['Kappa combined']):.4f} ({int(s['n compared']):,})",
                    "Paired kappa difference [95% CI]": dk})
t2 = pd.DataFrame(t2_rows)


#### Table 3 - paired change in kappa

In [ ]:
t3_rows = []
for v in ORDER:
    p = pa_i.loc[v]
    row = {"Variant": v, "Compared N": fmt_n(p["combined n"])}
    for key, name in [("urine", "Urine"), ("creatinine", "Creatinine"), ("combined", "Combined")]:
        o, lo, hi = p[f"dkappa {key}"], p[f"{key} CI low"], p[f"{key} CI high"]
        row[f"{name} change in kappa [95% CI]"] = fmt_ci(o, lo, hi)
        check(f"{v} {key}: point estimate inside its CI", (lo - 1e-9) <= o <= (hi + 1e-9))
    t3_rows.append(row)
t3 = pd.DataFrame(t3_rows)
check("Table 2 has V5 + all 23 variants", len(t2) == 24 and t2["Variant"].is_unique)
check("Table 3 lists exactly the variants with >=1 label changed",
      set(t3["Variant"]) == set(reg.loc[(reg["labels_changed_common"] > 0), "variant"]) - {"baseline"})
nochange = [v for v in VARIANTS if v not in T3_SET]
check("variants with no label change have a zero paired change (combined)",
      all(pa_i.loc[v, "dkappa combined"] == 0 for v in nochange))
if "urine_evaluable" in fz.columns:
    check("Evaluable N for V5 equals urine_evaluable in labels_v5",
          int(fz["urine_evaluable"].astype(bool).sum()) == int(p2_i.loc["baseline", "Evaluable N"]))


#### Supplementary tables

In [ ]:
t_s1 = pd.DataFrame({
    "Variant": [vname(v) for v in reg["variant"]], "Changed parameter": reg["changed_parameter"].values,
    "Cohort N": reg["cohort_n"].map(fmt_n).values, "Left cohort N": reg["left_cohort_n"].map(fmt_n).values,
    "Entered cohort N": reg["entered_cohort_n"].map(fmt_n).values, "Common N": reg["common_stays_n"].map(fmt_n).values,
    "Labels changed N": reg["labels_changed_common"].map(fmt_n).values,
    "Negative to positive": reg["neg_to_pos"].map(fmt_n).values,
    "Positive to negative": reg["pos_to_neg"].map(fmt_n).values,
    "Compared with reference N": reg["common_with_reference_n"].map(fmt_n).values})

stab = reg[reg["variant"] != "baseline"].copy()
stab["stab"] = (100 * (stab["common_stays_n"] - stab["labels_changed_common"]) / stab["common_stays_n"])
stab = stab.sort_values(["stab", "variant"])

In [ ]:
t_s2 = pd.DataFrame({
    "Variant": stab["variant"].values, "Common N": stab["common_stays_n"].map(fmt_n).values,
    "Labels changed N": stab["labels_changed_common"].map(fmt_n).values,
    "Negative to positive": stab["neg_to_pos"].map(fmt_n).values,
    "Positive to negative": stab["pos_to_neg"].map(fmt_n).values,
    "Label stability %": stab["stab"].map(lambda x: f"{x:.3f}").values})
check("Table S2 lists all 23 variants", len(t_s2) == 23)

In [ ]:


t_s3 = pd.DataFrame([
    {"Outcome": "Combined (any KDIGO criterion or CRRT)", "n compared": fmt_n(b2["n compared"]),
     "Kappa": f"{b2['Kappa combined']:.4f}", "TP": fmt_n(g["TP"]), "FP": fmt_n(g["FP"]), "FN": fmt_n(g["FN"]),
     "TN": fmt_n(g["TN"]), "Sensitivity": f"{g['sensitivity']:.3f}", "Specificity": f"{g['specificity']:.3f}",
     "PPV": f"{g['ppv']:.3f}", "NPV": f"{g['npv']:.3f}"},
    {"Outcome": "Creatinine criteria only", "n compared": fmt_n(b2["n compared"]), "Kappa": f"{b2['Kappa creatinine']:.4f}"},
    {"Outcome": "Urine-output criteria only", "n compared": fmt_n(b2["n compared"]), "Kappa": f"{b2['Kappa urine']:.4f}"},
]).fillna(NODATA)

In [ ]:
t_s4 = gs.copy()
t_s4["label"] = t_s4["label"].map(vname)
for c_ in ("TP", "FP", "FN", "TN"):
    t_s4[c_] = t_s4[c_].map(fmt_n)
for c_ in ("sensitivity", "specificity", "ppv", "npv", "kappa"):
    t_s4[c_] = t_s4[c_].map(lambda x: f"{x:.3f}")
t_s4 = t_s4.rename(columns={"label": "Variant", "sensitivity": "Sensitivity", "specificity": "Specificity",
                            "ppv": "PPV", "npv": "NPV", "kappa": "Kappa"})


In [ ]:
check("Table S3 TP+FP+FN+TN equals n compared", int(g["TP"] + g["FP"] + g["FN"] + g["TN"]) == int(b2["n compared"]))
check("n compared (summary) equals registry reference denominator", int(b2["n compared"]) == n_ref)
check("neg->pos + pos->neg equals labels changed",
      ((stab["neg_to_pos"] + stab["pos_to_neg"]) == stab["labels_changed_common"]).all())
check("labels changed never exceed common stays", (stab["labels_changed_common"] <= stab["common_stays_n"]).all())
check("V5 cohort = left + common for every variant", ((stab["left_cohort_n"] + stab["common_stays_n"]) == n_final).all())
check("variant cohort = entered + common for every variant",
      ((stab["entered_cohort_n"] + stab["common_stays_n"]) == stab["cohort_n"]).all())
check("cohort flow: total - early - no-evidence = analysis cohort", N_total - n_early - n_noev == n_final)
check("AKI % in summary equals AKI N / cohort N", abs(float(b2["AKI %"]) - 100 * n_aki / n_final) < 0.01)
check("AKI N / cohort N equals AKI % for every variant",
      ((100 * p2["AKI N"] / p2["Cohort N"] - p2["AKI %"]).abs() < 0.01).all())




In [ ]:
TABLES = {"table1_characteristics": t1, "table2_variants": t2, "table3_paired_dkappa": t3,
          "tableS1_variant_registry": t_s1, "tableS2_label_stability": t_s2,
          "tableS3_primary_vs_reference": t_s3, "tableS4_gap_sweep": t_s4}
for nm, df in TABLES.items():
    df.to_csv(D_TAB / f"{nm}.csv", index=False)
try:
    with pd.ExcelWriter(OUT / "tables.xlsx") as xw:
        for nm, df in TABLES.items():
            df.to_excel(xw, sheet_name=nm.split("_")[0][:31], index=False)
except ImportError:
    print("Error")

### Figures

In [ ]:
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#888888", "axes.labelcolor": "#222222", "xtick.color": "#444444",
                     "ytick.color": "#222222", "figure.dpi": 100})
C_MAIN, C_ACC, C_NEG, C_POS = "#0072B2", "#D55E00", "#56B4E9", "#E69F00"
GRID = "#E3E3E3"


def save(fig, folder, stem):
    fig.savefig(folder / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(folder / f"{stem}.svg", bbox_inches="tight")
    plt.close(fig)

#### Figure 2 (paired change in kappa (forest))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.5, 0.42 * len(ORDER) + 2.2), sharey=True)
panels = [("urine", "a. Urine-output component"), ("creatinine", "b. Creatinine component"), ("combined", "c. Combined")]
ypos = np.arange(len(ORDER))[::-1]
for ax, (key, title) in zip(axes, panels):
    ax.axvline(0, color="#555555", lw=1)
    ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
    for y, v in zip(ypos, ORDER):
        r = pa_i.loc[v]
        o, lo, hi = r[f"dkappa {key}"], r[f"{key} CI low"], r[f"{key} CI high"]
        if o == 0 and lo == 0 and hi == 0:
            ax.plot(0, y, marker="o", mfc="white", mec="#777777", mew=1.2, ms=6, ls="none", zorder=3)
        else:
            ax.plot([lo, hi], [y, y], color=C_MAIN, lw=2, solid_capstyle="round", zorder=2)
            ax.plot(o, y, marker="o", mfc=C_MAIN, mec="white", mew=1.0, ms=7, ls="none", zorder=3)
    ax.set_title(title, loc="left", fontsize=10, fontweight="bold")
    ax.set_xlabel("Paired change in kappa vs V5 (95% CI)")
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(5))     # avoids overlapping tick labels between panels
axes[0].set_yticks(ypos); axes[0].set_yticklabels(ORDER); axes[0].set_ylim(-0.7, len(ORDER) - 0.3)
handles = [Line2D([], [], marker="o", color=C_MAIN, lw=2, label="Paired change (95% CI)"),
           Line2D([], [], marker="o", mfc="white", mec="#777777", ls="none", label="No label change (kappa difference = 0)")]
fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, D_FIG, "fig2_forest")

#### maximum urine-output gap sweep (gap variants + V5 = no limit)

In [ ]:
gv = [v for v in reg["variant"] if v.startswith("gap_") and v.endswith("h") and v[4:-1].isdigit()]
gv = sorted(gv, key=lambda v: int(v[4:-1]))
gx = [f"{int(v[4:-1])} h" for v in gv] + ["none\n(V5)"]
prev = [float(p2_i.loc[v, "AKI %"]) for v in gv] + [float(b2["AKI %"])]
chg = [float(chg_pct[v]) for v in gv] + [0.0]
figg, (ga, gb) = plt.subplots(1, 2, figsize=(7.2, 3.2))
for ax_, vals, colr, yl in [(ga, prev, C_MAIN, "AKI prevalence (%)"), (gb, chg, C_ACC, "Labels changed vs V5 (%)")]:
    ax_.plot(range(len(vals)), vals, "-o", color=colr, lw=2, ms=6)
    for i_, x_ in enumerate(vals):
        ax_.annotate(f"{x_:.1f}", (i_, x_), textcoords="offset points",
                     xytext=(14, -12) if (ax_ is ga and i_ == 0) else (0, 8), ha="center", fontsize=8)
    ax_.set_xticks(range(len(vals))); ax_.set_xticklabels(gx); ax_.set_xlabel("Maximum urine-output gap"); ax_.set_ylabel(yl)
figg.tight_layout()
save(figg, D_FIG, "figS1_gap_sweep")

In [ ]:
C_LEFT, C_ENTER, C_N2P, C_P2N = "#CC79A7", "#009E73", "#E69F00", "#56B4E9"
d = reg[reg["variant"] != "baseline"].copy()
d["left_pct"] = 100 * d["left_cohort_n"] / n_final
d["enter_pct"] = 100 * d["entered_cohort_n"] / n_final
d["n2p_pct"] = 100 * d["neg_to_pos"] / d["common_stays_n"]
d["p2n_pct"] = 100 * d["pos_to_neg"] / d["common_stays_n"]
d["total"] = d["left_pct"] + d["enter_pct"] + d["n2p_pct"] + d["p2n_pct"]
d = d.sort_values("total")
y = np.arange(len(d))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 7.2), sharey=True)
a1.barh(y, -d["left_pct"], height=0.6, color=C_LEFT); a1.barh(y, d["enter_pct"], height=0.6, color=C_ENTER)
a1.axvline(0, color="#555555", lw=1)
a1.set_title("a. Cohort composition vs V5", loc="left", fontsize=10, fontweight="bold")
a1.set_xlabel(f"% of the V5 cohort ({n_final:,} stays)")
a1.legend(handles=[Patch(color=C_LEFT, label="Left the cohort"), Patch(color=C_ENTER, label="Entered the cohort")],
          frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2)
a1.set_xticks(a1.get_xticks()); a1.set_xticklabels([f"{abs(t):g}" for t in a1.get_xticks()])
a2.barh(y, d["n2p_pct"], height=0.6, color=C_N2P); a2.barh(y, -d["p2n_pct"], height=0.6, color=C_P2N)
a2.axvline(0, color="#555555", lw=1)
a2.set_title("b. Label change on stays common to V5 and the variant", loc="left", fontsize=10, fontweight="bold")
a2.set_xlabel("% of common stays")
a2.legend(handles=[Patch(color=C_P2N, label="Positive to negative"), Patch(color=C_N2P, label="Negative to positive")],
          frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2)
a2.set_xticks(a2.get_xticks()); a2.set_xticklabels([f"{abs(t):g}" for t in a2.get_xticks()])
for ax in (a1, a2):
    ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
a1.set_yticks(y); a1.set_yticklabels(d["variant"]); a1.set_ylim(-0.7, len(d) - 0.3)
fig.text(0.01, 0.005, "Label change is measured only on stays present in both the V5 and the variant cohort.",
         fontsize=8, color="#444444")
fig.tight_layout(rect=(0, 0.04, 1, 1))
save(fig, D_FIG, "figS2_decomposition")
